# Altair Practice Lab

In this assignment we will be crafting a series of visualizations using
Altair to get practice working with real data in this context.

Your responses should be within the functions given,
using appropriate helper functions to help with clarity
and reduce redundancy.

## Rubric

The criteria to receive full credit on this assignment is a good-faith attempt at each portion.

A good-faith attempt should either:

- be fundamentally correct producing the expected output with minimal deviations,
- **OR** contain an explanation of what does not work and _details on what was tried_.

**For half credit, at least half of the assignment should have good-faith attempts.**

Your charts do not need to match the examples exactly!  They are helpful to get a sense of what you're after, but focus on the problem description and feel free to experiment as you see fit.

## Introducing the Dataset

In the data directory you'll find three files:

**legislators.csv** which consists of ~7400 records representing state legislators, it has the following fields:

- name
- given_name
- family_name
- party: As reported by the state.
- gender: Male / Female / Other\*
- jurisdiction: This field contains an identifier for the state or jurisdiction, see below for details.
- district: the name of the district represented
- type: upper | lower - The classification of the legislative. Most states have both, but DC and NE only have an upper chamber.

Note: Accurate data on gender is hard to come by in many states. There may be irregularities in this field. This is also why this field does not make further distinctions beyond Male/Female/Other.

**populations.csv** 

A CSV file with a row with each state's population.  

**actions.csv**

A CSV file with legislative actions for a few states across a range of years:

- description: action text as reported by state
- date: date of action recorded
- classification: not used for this assignment
- state: 2-letter state postal code (note: these are upper case, other files use lower case)
- session: an identifier representing which session the action comes from (not used)

In [2]:
# PUT YOUR IMPORTS HERE

import altair as alt
import pandas as pd
import numpy as np

## Part 1: Data Exploration

First, we'll build a few exploratory visualizations to get a sense of the data for this assignment.

### 1.0: Cleaning

As mentioned above, there is no 'state' field. Jurisdiction is in the format:

`ocd-jurisdiction/country:us/state:nc/government`

and for non-states:

- `ocd-jurisdiction/country:us/district:dc/government`
- `ocd-jurisdiction/country:us/territory:pr/government`

So for our purposes, we want to add a `state` column from the two letter code after either "state:", "district:", or "territory:".
(We will treat DC and PR as states.)

Complete the function `legislators_df` which should return the data from `legislators.csv` in a dataframe, with an additional `state` column.

In [ ]:
def legislators_df():
    legislators = pd.read_csv("data/legislators.csv")

    legislators["state"] = legislators["jurisdiction"].str.extract(r"/(?:[^/:]+):([^/]+)/government$")

    return legislators

# render dataframe
legislators_df()

### 1.1: Initial Plot

First let's build a visualization of gender breakdowns in state legislatures.
Use the following:

- stacked bars per state
- each segment of stacked bar is gender

Your graph should somewhat resemble *imgs/ex1.1.png*.

In [3]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [ ]:
def states_by_gender_initial(df):
    state_by_gender_chart = alt.Chart(df).mark_bar().encode(
        x=alt.X("state:N", title="State"),
        y=alt.Y("count():Q", title="Number of Legislators"),
        color=alt.Color("gender:N", title="Gender")
    )

    return state_by_gender_chart

#render chart
states_by_gender_initial(legislators_df())


### 1.2: Improvements

While it is clear from the first chart that there are more elected officials that are men than women, it is hard to compare across states.

Make the following adjustments:

- Normalize the chart so that each bar is a percentage, allowing for direct comparison across states.
- Since this is US political data, the colors red and blue have a strong meaning, associated with the Republican and Democratic parties. Change the color scheme to avoid red and blue. (I chose #8624f5 for women and #1fc3aa for men based on this article: <https://blog.datawrapper.de/gendercolor/>)
- Two states are very close to 50%, add a line at 50% using a layered chart to make it easier to see if they exceed 50% or not.

Your graph should somewhat resemble *imgs/ex1.2.png*.

In [ ]:
def states_by_gender_improved(df):
    gender_categories = ["Female", "Male", "Other"]
    custom_colors = ["#8624f5", "#1fc3aa", "#ff5964"]

    state_by_gender_chart = alt.Chart(df).mark_bar().encode(
            x=alt.X("state:N", title="State"),
            y=alt.Y("count():Q", stack="normalize", title="Percentage of Legislators"),
            color=alt.Color("gender:N", title="Gender", scale=alt.Scale(domain=gender_categories, range=custom_colors))
        )

    fifty_percent_line_chart = alt.Chart().mark_rule(color="#808080").encode(
        y=alt.datum(0.5)
    )

    final_chart = state_by_gender_chart + fifty_percent_line_chart
    
    return final_chart

# render chart
states_by_gender_improved(legislators_df())

## Part 2: Party Breakdown

We'll now take a look at party control. We can start with essentially the same chart.

### 2.0 - Party Control

Copy your code from 1.2 above & modify it to use party instead of gender.  Your graph will wind up with too many parties, see `imgs/ex2.0.png`.

In [ ]:
def party_control_raw(df):
    party_control_chart = alt.Chart(df).mark_bar().encode(
            x=alt.X("state:N", title="State"),
            y=alt.Y("count():Q", stack="normalize", title="Percentage of Legislators"),
            color=alt.Color("party:N", title="Party",)
        )
    
    return party_control_chart

party_control_raw(legislators_df())

### 2.1 - Cleaning Data

The above graph still has some shortcomings:

- Most states have an upper and lower chamber, and party control may vary between them. We'll need to make two bars per state (which we'll tackle in 2.2).
- Also, there are too many variations of party as you can see here:

Let's transform the data again, adding a new column "party_code" with the following rules:

- if the word 'Democratic' appears, set party_code to 'D'
- if the word 'Republican' appears, set the party_code to 'R'
- otherwise, set the party_code to 'O'

Party data in NE, DC, and PR does not work with this scheme.
For simplicity, we will exclude them from our analysis.

For this portion, implement `clean_party_df` which should return a modified legislators DataFrame with the `party_code` column, and the rows for states 'DC', 'NE' and 'PR' dropped.

In [ ]:
def clean_party_df():
    # start with the DataFrame from part 1 & return transformed copy
    df = legislators_df()

    # remove DC, NE, and PR
    drop_states = ["dc", "ne", "pr"]
    df = df[~df['state'].isin(drop_states)]

    party_conditions = [
        df["party"].str.contains("Democratic"),
        df["party"].str.contains("Republican")
    ]
    party_abbreviations = ["D", "R"]
    df["party_code"] = np.select(party_conditions, party_abbreviations, default="O")

    return df

clean_party_df()

### 2.2 - Faceted Plot

Add a function `party_control_by_chamber` that contains the following elements:

- One bar per state, **along the Y axis**.
- Each bar should consist of a stack: a blue portion, a green portion, and a red portion, corresponding to the D, O, and R `party_code` respectively.
- A vertical line at the 50% mark, indicating (likely) party control.
- Finally, facet the chart on `type` so that you get a set of bars for the lower and upper chambers.

See `imgs/ex2.2.png` for an example.

In [ ]:
def party_control_by_chamber(df):
    party_codes = ["D", "O", "R"]
    custom_colors = ["#9dd8e7", "#7ceab3", "#7f1414"]
    
    party_control_by_chamber_chart = alt.Chart(df).mark_bar().encode(
        x=alt.X("count():Q", stack="normalize", title="Percentage of Legislators"),
        y=alt.Y("state:N"),
        color=alt.Color("party_code:N", scale=alt.Scale(domain=party_codes, range=custom_colors))
    ).facet(
        column="type:N"
    )

    return party_control_by_chamber_chart

party_control_by_chamber(clean_party_df())

## Part 3: Comparing by Population

For part three, we are interested in the relationship of various properties of legislatures to the total population of the state.

To do this, we'll need to create a combined DataFrame that mixes in data from `populations.csv`.

### 3.0 - Create Combined DataFrame

Write the function `population_combined_df`, which should return a DataFrame with the columns:

- state: abbreviation of state
- upper: total seats in upper chamber
- lower: total seats in lower chamber
- pop_2020: the 2020 population, obtained from merging with `population.csv`

**Data Note:** These numbers are based on the non-vacant seats as-of a particular day in September 2024. Vacancies will cause the counts to be off by a bit, but the general size of the legislature should be roughly the same.

In [ ]:
def population_combined_df():
    legislators_df = clean_party_df()
    legislators_df = pd.crosstab(legislators_df["state"], legislators_df["type"]).reset_index()

    pop_df = pd.read_csv("data/populations.csv")
    
    population_combined_df = pd.merge(
        legislators_df,
        pop_df,
        left_on="state",
        right_on="state",
        how="inner"
    )

    return population_combined_df

population_combined_df()

### 3.1 - Create Population vs. Seats Scatterplot

Create a new plot with two layers:

- Population on the X axis
- Number of seats on the Y axis
- Upper chamber points should be purple and use the 'triangle-up' shape.
- Lower chamber points should be orange and use the 'triangle-down' shape.
- Make a customization or two to your chart's default labels and axes, whatever you feel is appropriate.

Hint: You can layer two charts for this.

In [ ]:
def scatter_pop_size():
    data = population_combined_df()

    upper_scatter_plot = alt.Chart(data).mark_point().encode(
        x=alt.X("pop_2020:Q", title="Population"),
        y=alt.Y("upper:Q", title="Chamber Seats"),
        shape=alt.value("triangle-up"),
        color=alt.value("purple")
    )

    lower_scatter_plot = alt.Chart(data).mark_point().encode(
        x=alt.X("pop_2020:Q", title="Population"),
        y=alt.Y("lower:Q", title="Chamber Seats"),
        shape=alt.value("triangle-down"),
        color=alt.value("orange")
    )

    final_chart = lower_scatter_plot + upper_scatter_plot

    return final_chart

scatter_pop_size()

### 3.2 - Regressions

Add two more layers, a purple & orange regression line for each chamber.  See `imgs/ex3.2.png`

Hint: See `transform_regression`.

In [ ]:
def scatter_pop_size_regression():
    data = population_combined_df()
    
    upper_scatter_plot = alt.Chart(data).mark_point().encode(
        x=alt.X("pop_2020:Q", title="Population"),
        y=alt.Y("upper:Q", title="Chamber Seats"),
        shape=alt.value("triangle-up"),
        color=alt.value("purple")
    )

    upper_regression = upper_scatter_plot.transform_regression("pop_2020", "upper").mark_line()

    lower_scatter_plot = alt.Chart(data).mark_point().encode(
        x=alt.X("pop_2020:Q", title="Population"),
        y=alt.Y("lower:Q", title="Chamber Seats"),
        shape=alt.value("triangle-down"),
        color=alt.value("orange")
    )

    lower_regression = lower_scatter_plot.transform_regression("pop_2020", "lower").mark_line()


    final_chart = lower_scatter_plot + lower_regression + upper_scatter_plot + upper_regression

    return final_chart

scatter_pop_size_regression()

## Part 4: Actions Heatmap

The file `actions_il-in-mi-wi_2021-2024.csv` contains nearly half a million records, representing every official action taken on a piece of legislation in these four midwestern states over the past two sessions.

Legislatures work quite differently, some meet all year, while others meet for very short periods.
By creating a heatmap of what days actions take place, we can get a sense of how different states compare.

### 4.0 - Load Actions

Complete `actions_df`, which should load the data from `actions_il-in-mi-wi_2021-2024.csv`.

Tips: 
- Make sure that the `date` column is loaded as a date type!
- Dates are in YYYY-MM-DD format, though some also have additional characters for time, which you will want to ignore.

In [ ]:
def actions_df():
    df = pd.read_csv("data/actions_il-in-mi-wi_2021-2024.csv")

    df["date"] = pd.to_datetime(df["date"], format="mixed", utc=True)

    # remove the time
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

    return df

actions_df()

### 4.1 - Actions Heatmap

Generate a heatmap (using `mark_rect`) with:

- a row per state
- each row consists of shaded marks with shading based on the total action count for a given week

Tip: Use the 'yearweek(date)' aggregation for the X channel.

See `imgs/ex4.1.png`.

In [ ]:
def actions_heatmap_scaled(df):
    actions_heatmap_chart = alt.Chart(df).mark_rect().encode(
        x=alt.X("yearweek(date):T"),
        y=alt.Y("state:N"),
        color=alt.Color("count():Q")
    )

    return actions_heatmap_chart

actions_heatmap_scaled(actions_df())

### 4.2 - Excluding IL Outliers

Illinois clearly dominates the above graph, below modify two calls to `actions_heatmap` with a modified dataframe that excludes IL, and a modified dataframe that only includes IL.

(Note how by using functions in our dataframe we can more easily reuse portions by making small adjustments to the data.)

See `ex4.2a.png` and `ex4.2b.png`

In [ ]:
# MODIFY THIS LINE (exclude IL)
full_df = actions_df()
no_il_df = full_df[full_df["state"] != "IL"]
actions_heatmap_scaled(no_il_df)

In [ ]:
# MODIFY THIS LINE (only IL)
full_df = actions_df()
only_il_df = full_df[full_df["state"] == "IL"]
actions_heatmap_scaled(only_il_df)

#### 4.3 - Cumulative Line Chart

Another way to view this data would be with a cumulative line chart.

Create a chart with:

- days on the X axis
- cumulative actions to date on the Y axis
- one line per state

Hint: To do this you will need to look at the `transform_window` function.

See `ex4.3.png` for an example.

In [16]:
def actions_cumulative(df):
    cumulative_actions_chart = alt.Chart(df).mark_line().encode(
        x=alt.X("yearmonthdate(date):T"),
        y=alt.Y("count():Q"),
        color=alt.Color("state:N")
    )

actions_cumulative(actions_df())

## Part 5. Replicating a Chart

Find a chart (perhaps the one you're using for your first Analysis post!) and do your best to replicate it with Altair. Since you likely won't have access to the raw data-- start by mocking up a dataset that is similar enough to what your chart uses, don't worry about exact values, just that the correct attributes are present with some variation.

Your replicated chart should use similar colors, labels, etc. to the one that you chose to reproduce. Again, we aren't looking for a pixel-perfect replica, but something that pushes you to explore parts of Altair's documentation/features that you didn't use above.


In [3]:
cost_of_living_per_dollar_index_df = pd.DataFrame({
    "City": ["Tuscon", "Jacksonville", "San Antonio", "Omaha", "Kansas City", "Albuquerque", "Raleigh", "Austin", "Tampa", "Houston", "Oklahoma City", "Colombus", "Indianapolis", "Charlotte", "Milwaukee", "Fort Worth", "Colorado Springs", "Nashville", "Minneapolis", "Atlanta", "Dallas", "Sacramento", "San Diego", "Baltimore", "Portland", "Denver", "Las Vegas", "Boston", "New Orleans", "Phoenix", "Philadelphia", "Chicago", "Detroit", "Oakland", "Seattle", "Washington DC", "San Jose", "Miami", "San Francisco", "Los Angeles", "New York"],
    "Quality of life per dollar": [300, 297, 295, 288, 286, 285, 283, 281, 277, 276, 273, 272, 271, 271, 270, 268, 258, 256, 251, 244, 242, 236, 236, 236, 235, 228, 228, 225, 216, 215, 213, 213, 210, 210, 205, 200, 198, 192, 182, 157, 134],
    "Region": ["West", "South", "South", "Midwest", "Midwest", "West", "South", "South", "South", "South", "South", "Midwest", "Midwest", "South", "Midwest", "South", "West", "South", "Midwest", "South", "South", "West", "West", "South", "West", "West", "West", "Northeast", "South", "West", "Northeast", "Midwest", "Midwest", "West", "West", "South", "West", "South", "West", "West", "Northeast"]
})

# created a sorted list of cities to prevent sort confusion withing alt.encode()
sorted_df = cost_of_living_per_dollar_index_df.sort_values(
    by="Quality of life per dollar", 
    ascending=False
)
ordered_cities = sorted_df["City"].tolist()

In [ ]:
regions = ["Northeast", "Midwest", "South", "West"]
custom_colors = ["#4b59db", "#dbc34b", "#db614b", "#4bdb9f"]

base_chart = alt.Chart(cost_of_living_per_dollar_index_df).encode(
    x=alt.X("Quality of life per dollar:Q"),
    y=alt.Y("City:N", sort=ordered_cities),
    color=alt.Color("Region:N", scale=alt.Scale(domain=regions, range=custom_colors), legend=alt.Legend(orient="top", direction="horizontal", title=None)),
).properties(
    title=alt.TitleParams(
        text="US cities",
        subtitle="ranked by quality of life per dollar",
        anchor="start",
        fontSize=18
    )
)

bar_chart = base_chart.mark_bar(size=1)
circle_chart = base_chart.mark_circle(size=100)

bar_chart + circle_chart